<!-- While the ingestion is running (or after it finishes), create another notebook 

Comparing the two approaches

With minsearch (single process):

Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps

With sqlitesearch (two processes):

Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready 


For larger production systems, use the same pattern with a different backend:

    Elasticsearch
    OpenSearch
    Qdrant (vector database)
    Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

-->


In [ ]:
from sqlitesearch import TextSearchIndex

#Connect to the same database:

sqlite_index = TextSearchIndex(
    text_fields = ['question', 'answer', 'section'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [8]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import os

# Create the OpenAI client from the environment variable
from openai import OpenAI
openai_client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

In [ ]:
# We use the RAGBase class from rag_helper.py with this sqlitesearch index.

# Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

from rag_helper import RAGBase

assistant = RAGBase(sqlite_index, openai_client)

In [10]:
assistant.rag("I just discovered the course, can I still join?")

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [11]:
# Cleaning up

# When you're done, close the database connection:

sqlite_index.close()